In [1]:
import sys
sys.path.append("..")   # le notebook est dans notebooks/, la racine du projet est un cran au-dessus

from src.data_loader import get_prepared_dataset

stations, trajets, abonnes = get_prepared_dataset()

In [2]:
t8 = trajets[trajets["heure"] == 8]
print("Trajets à 8 h :", len(t8), "sur", len(trajets))
print(trajets["heure"].value_counts().sort_index())
print(trajets["jour"].unique())

Trajets à 8 h : 61 sur 692
heure
5      8
6     23
7     49
8     61
9     51
10    26
11    25
12    40
13    35
14    46
15    55
16    73
17    60
18    46
19    32
20    27
21    16
22    10
23     9
Name: count, dtype: int64
<StringArray>
['ven', 'lun', 'dim', 'mar', 'jeu', 'mer', 'sam']
Length: 7, dtype: str


In [3]:
comptes = t8.groupby("station_depart").size().reindex(stations["station_id"], fill_value=0)
print(comptes.value_counts().sort_index())
print("moyenne par station :", comptes.mean())

0    86
1    47
2     7
Name: count, dtype: int64
moyenne par station : 0.4357142857142857


In [4]:
tab = stations.set_index("station_id")[["type_quartier"]].copy()
tab["departs_8h"] = comptes
print(tab.groupby("type_quartier")["departs_8h"].agg(["count", "sum", "mean"]))

               count  sum      mean
type_quartier                      
bureaux           51   19  0.372549
gare              21    8  0.380952
résidentiel       39   22  0.564103
université        29   12  0.413793


In [5]:
print("moyenne :", comptes.mean())
print("variance :", comptes.var())

moyenne : 0.4357142857142857
variance : 0.3483556012332991


Conclusion de l'exploration (Q3).
À 8 h, il y a 61 trajets sur 692, soit en moyenne 0,44 départ par station. 86 stations sur 140 (61 %) n'ont aucun départ à 8 h, et aucune n'en a plus de 2. Un simple comptage par station ne suffit donc pas : il ne donnerait que 0, 1 ou 2 et reflèterait surtout le hasard. Les quartiers résidentiels ont un peu plus de départs par station (0,56 contre 0,37 à 0,41 ailleurs), mais l'écart repose sur très peu de trajets et reste à confirmer. Les départs sont de petits entiers avec beaucoup de zéros, et la variance (0,35) est proche de la moyenne (0,44) : une régression de Poisson est adaptée.